# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Basil-Maqbool/flyrank-internship-assignment1/blob/main/work/notebooks/w01_research_question.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
# Replace this with your exact repository URL
REPO_URL = "https://github.com/Basil-Maqbool/flyrank-internship-assignment1"
REPO_DIR = "flyrank-internship-assignment1"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    # This navigates to the correct subdirectory inside the cloned repo
    os.chdir("work/notebooks")

print("Current Working Directory:", os.getcwd())

Current Working Directory: c:\Users\Basil\Desktop\flyrank-internship-assignment1\work\notebooks


**Lane: Content Refresh and Opportunity Scoring**

I chose this specific track because the provided starter pipeline already demonstrates a massive 3x performance lift over a manual heuristic. Specifically, the random forest model hits a Precision@50 of 0.740, while the simple baseline only achieves 0.240 (12 correct vs 37 correct out of 50). This gap indicates that a learned ranking system is highly effective at triaging declining pages. Furthermore, the starter provides a clear roadmap for upgrading the proxy label (current window decline) to a more robust, future-facing target (predicting the next 30 days based on the previous 90 days).

I am choosing Lane 2: The Content Refresh Queue (Search Performance Decay & Optimization).
Organic search traffic is the lifeblood of our client digitl footprint/presence.
However, search rankings are not static; content decays over time as search intent shifts,
competitors publish fresher resources, or search algorithms update. Identifying which pages are
experiencing a genuine structural decline versus normal daily traffic fluctuations is a massive challenge.
By focusing on this lane, we can move from reactive, ad-hoc content updates to a proactive, machine-learning-driven queue.
This will help our content teams prioritize their limited editing hours on the pages that have the highest probability of recovering significant lost traffic.

- **Decision to optimize:** Identifying exactly which declining pages a content team should prioritize for a refresh.
- **Actor:** Content editors and SEO strategists who operate under a strict weekly bandwidth (e.g., 20-50 pages).
- **Action:** They will evaluate the top-ranked pages from the queue alongside the provided reason codes, deciding whether to execute a rewrite, update metadata, or just monitor the page.
- **Cost of errors:**
  - *False Positive:* Wasted human effort. The editor spends 15+ minutes reviewing a page that didn't actually need intervention, draining resources from true issues.
  - *False Negative:* Traffic bleeds out. A genuinely decaying page is ignored, and the lost opportunity compounds week over week.
- **Why ML is necessary:** The indicators of decline are scattered across multiple dimensions (age, impressions, rank, engagement). Simple thresholds fail to capture these non-linear relationships (as shown by the baseline's poor 12/50 score). A machine learning model successfully synthesizes these signals into a reliable ranking.

The Search Question: "Which of our high-traffic pages are experiencing a structural decline in organic search performance and should be prioritized for a content refresh?"
The Unit of Analysis: A single page (URL) over a 90-day window.
The Model's Output: A probability score (between 0.0 and 1.0) indicating the likelihood that a page is in a structural downward trend, sorted as a ranked queue.
The Business Decision: "Which pages should our editorial and SEO team invest time and resources into rewriting this week?"
The Concrete Action: The content team opens the top-ranked pages from our queue, runs a gap analysis against current search intent, refreshes the copy, and redeploys the page.
The Cost of a Wrong Recommendation (False Positive): If the model flags a page as declining when it is actually stable, we waste valuable writer hours (typically costing $150 - $300 per page in labor) modifying content that didn't need to be touched. Worse, we risk disrupting a high-performing page and causing a real drop in rankings.
The Cost of a Missed Opportunity (False Negative): If the model fails to flag a page that is actively decaying, the page will continue its downward slide in search results. Over months, this compounds into thousands of lost visits, lower conversion rates, and direct revenue loss.
Why ML is needed: A human editor cannot manually monitor thousands of pages across multiple metrics (impressions, CTR, positions, age, trend direction) simultaneously. Hand-written rules (like "refresh if older than 180 days") are too simple and generate too many false positives. ML can analyze complex, non-linear relationships across these variables to surface the true structural declines.

**1. High prevalence of decline:** Approximately 54.2% (16,262) of the pages in this slice are flagged with a downward trend, providing ample signal for the model to learn from.
    
**2. Massive performance gap:** The hand-coded heuristic only identifies 12 true decliners in the top 50 (Precision@50 = 0.240), whereas the random forest model successfully finds 37 (Precision@50 = 0.740).

**3. Meaningful traffic at stake:** The median impressions per page sit at 731 across the 32 clients. This means the pages we are triaging represent genuine search visibility worth recovering.

In [2]:
import pandas as pd
import json, os

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
print(f'Total observations: {len(df):,}')
print(f'Unique clients: {df.client_id.nunique()}')
print(f'Proportion of declining pages: {(df.trend_direction.str.lower() == "down").mean():.3f}')
print(f'Median 90d impressions: {df.impressions_90d.median():.0f}')
print()

# Load results if available (generated by running the pipeline scripts first)
results_path = '../../outputs/model_results.json'
if os.path.exists(results_path):
    results = json.load(open(results_path))
    print('Validation Precision@50 metrics:')
    for model_name, metrics in results['models'].items():
        print(f'  {model_name}: {metrics["precision_at_50"]:.3f}')
    print(f'  heuristic baseline: {results["baseline"]["baseline_precision_at_50"]:.3f}')
    print(f'Top performer: {results["best_model"]["name"]}')
else:
    print('Note: run scripts/run_all.py first to generate model_results.json')
    print('Indicative results from starter pipeline: Random Forest P@50 = 0.74, Baseline P@50 = 0.24')


Total observations: 30,000
Unique clients: 32
Proportion of declining pages: 0.542
Median 90d impressions: 731

Validation Precision@50 metrics:
  decision_tree: 0.620
  logistic_regression: 0.400
  random_forest: 0.740
  heuristic baseline: 0.240
Top performer: random_forest


From our exploratory analysis of the starter dataset, three key numbers stand out and justify this lane choice:
High Target Prevalence: 54.2% of the pages in our dataset are actively in a downward trend. This confirms we have a balanced classification problem with a substantial group of decaying pages to identify.
Massive Scale of Staleness: 0.6% of the pages have not been updated in over 180 days. However, the high decline rate across the dataset means many pages are decaying regardless of staleness — a simple rule-based approach would miss most declining pages.
High-Exposure Focus Group: There are stale pages that still pull in over 1,000 impressions every 90 days. These high-exposure, aging pages represent the exact high-impact "low hanging fruit" that a machine learning model can rank to maximize traffic recovery.


**What I can confidently claim:**
- The data shows a measurable, directional relationship between observed metrics (like impressions, age, and position) and the likelihood of a page declining.
- Based on the starter dataset, a tree-based model can rank decaying pages significantly better than a hardcoded rule.
- Generating automated reason codes alongside the ranking creates a practical, decision-support tool for editors.

**What I cannot claim:**
- **Causality:** I cannot guarantee that refreshing a flagged page will reverse the traffic loss (this requires A/B testing).
- **Algorithmic certainty:** I am not reverse-engineering Google's core algorithm; I am simply predicting based on observable trailing metrics.
- **Future guarantees:** The current target is a proxy (current window decline). The results on this 30k sample may drift when scaled to the full 79M row warehouse.

In organic search engine optimization, scientific humility is critical.
What we CANNOT claim: We cannot claim that our model "predicts Google's ranking algorithm" or that our features represent direct ranking factors. Google's core algorithm is a proprietary black box with thousands of real-time signals.
What we CAN claim: We are building a decision-support tool. We can claim that our model identifies historical statistical associations between observable page metrics (like content age, impressions, and CTR) and subsequent performance decay. The output is a directional, prioritized recommendation queue designed to optimize internal resource allocation, not a simulation of search engine mechanics.

## The one-paragraph frame

For content and SEO teams deciding which pages to refresh first, we will build a **ranked priority queue** from **historical search performance data** (~30,000 pseudonymized content items), predicting **content decay** (measured by trailing 90-day trend direction) at **Precision@50**. A wrong call costs **$150–$300 in wasted editor labor** per page, or missed decay compounds into thousands of lost visits. A plain rule isn't enough because **decay is driven by non-linear interactions across age, impressions, and update recency that fixed thresholds cannot capture**. We will claim only **observed, directional, decision-support** results.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.